# 00 - Setup & Configuration
This notebook sets up schemas, paths, and runtime variables for the Food Inspection project.

Medallion layers are separated by schema for Delta tables and by folder path for Parquet snapshots.

In [0]:
%pip install databricks-labs-dqx

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.removeAll()
print("All widgets cleared.")


All widgets cleared.


## Widget Parameters

In [0]:
dbutils.widgets.text("raw_path", "/Volumes/workspace/food_inspection/raw_data", "Raw Data Path")
dbutils.widgets.text("chicago_file", "Food_Inspections_20260411.csv", "Chicago CSV Filename")
dbutils.widgets.text(
    "dallas_file",
    "Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv",
    "Dallas CSV Filename"
)

dbutils.widgets.text("bronze_schema", "foodinspection_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "foodinspection_silver", "Silver Schema")
dbutils.widgets.text("gold_schema", "foodinspection_gold", "Gold Schema")

dbutils.widgets.text(
    "bronze_path",
    "/Volumes/workspace/foodinspection_bronze/food_inspection",
    "Bronze Snapshot Path"
)
dbutils.widgets.text(
    "silver_path",
    "/Volumes/workspace/foodinspection_silver/food_inspection",
    "Silver Snapshot Path"
)
dbutils.widgets.text(
    "gold_path",
    "/Volumes/workspace/foodinspection_gold/food_inspection",
    "Gold Snapshot Path"
)

dbutils.widgets.text("snapshot_run_id", "", "Snapshot Run Id (optional)")

print("raw_path    =", dbutils.widgets.get("raw_path"))
print("bronze_path =", dbutils.widgets.get("bronze_path"))
print("silver_path =", dbutils.widgets.get("silver_path"))
print("gold_path   =", dbutils.widgets.get("gold_path"))
print("bronze_schema =", dbutils.widgets.get("bronze_schema"))
print("silver_schema =", dbutils.widgets.get("silver_schema"))
print("gold_schema   =", dbutils.widgets.get("gold_schema"))

raw_path    = /Volumes/workspace/food_inspection/raw_data
bronze_path = /Volumes/workspace/foodinspection_bronze/food_inspection
silver_path = /Volumes/workspace/foodinspection_silver/food_inspection
gold_path   = /Volumes/workspace/foodinspection_gold/food_inspection
bronze_schema = foodinspection_bronze
silver_schema = foodinspection_silver
gold_schema   = foodinspection_gold


In [0]:
RAW_PATH = dbutils.widgets.get("raw_path")
CHICAGO_FILE = dbutils.widgets.get("chicago_file")
DALLAS_FILE = dbutils.widgets.get("dallas_file")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
BRONZE_PATH = dbutils.widgets.get("bronze_path")
SILVER_PATH = dbutils.widgets.get("silver_path")
GOLD_PATH = dbutils.widgets.get("gold_path")
SNAPSHOT_RUN_ID = dbutils.widgets.get("snapshot_run_id")

required_paths = {
    "raw_path": RAW_PATH,
    "bronze_path": BRONZE_PATH,
    "silver_path": SILVER_PATH,
    "gold_path": GOLD_PATH,
}

missing_paths = [name for name, value in required_paths.items() if value.strip() == ""]
if missing_paths:
    raise ValueError(
        "Missing required widget path values: "
        + ", ".join(missing_paths)
        + ". Set these widgets before running the notebook."
    )

if SNAPSHOT_RUN_ID == "":
    SNAPSHOT_RUN_ID = spark.sql(
        "SELECT date_format(current_timestamp(), 'yyyyMMdd_HHmmss') AS run_id"
    ).collect()[0]["run_id"]

snapshot_parts = spark.sql(
    """
    SELECT
      date_format(current_timestamp(), 'yyyy') AS snapshot_year,
      date_format(current_timestamp(), 'MM') AS snapshot_month,
      date_format(current_timestamp(), 'dd') AS snapshot_day
    """
).collect()[0]
SNAPSHOT_YEAR = snapshot_parts["snapshot_year"]
SNAPSHOT_MONTH = snapshot_parts["snapshot_month"]
SNAPSHOT_DAY = snapshot_parts["snapshot_day"]

RAW_CHICAGO_PATH = f"{RAW_PATH}/{CHICAGO_FILE}"
RAW_DALLAS_PATH = f"{RAW_PATH}/{DALLAS_FILE}"
RAW_SNAPSHOT_PATH = f"{RAW_PATH}/parquet_snapshots"

# Backward-compatible aliases for older notebook code
VOLUME_PATH = RAW_PATH
DATABASE_NAME = GOLD_SCHEMA

print(f"Raw Path         : {RAW_PATH}")
print(f"Raw Chicago Path : {RAW_CHICAGO_PATH}")
print(f"Raw Dallas Path  : {RAW_DALLAS_PATH}")
print(f"Bronze Schema    : {BRONZE_SCHEMA}")
print(f"Silver Schema    : {SILVER_SCHEMA}")
print(f"Gold Schema      : {GOLD_SCHEMA}")
print(f"Bronze Path      : {BRONZE_PATH}")
print(f"Silver Path      : {SILVER_PATH}")
print(f"Gold Path        : {GOLD_PATH}")
print(f"Raw Snapshot Path: {RAW_SNAPSHOT_PATH}")
print(f"Snapshot Run Id  : {SNAPSHOT_RUN_ID}")
print(f"Snapshot Date    : {SNAPSHOT_YEAR}/{SNAPSHOT_MONTH}/{SNAPSHOT_DAY}")

Raw Path         : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Bronze Schema    : foodinspection_bronze
Silver Schema    : foodinspection_silver
Gold Schema      : foodinspection_gold
Bronze Path      : /Volumes/workspace/foodinspection_bronze/food_inspection
Silver Path      : /Volumes/workspace/foodinspection_silver/food_inspection
Gold Path        : /Volumes/workspace/foodinspection_gold/food_inspection
Raw Snapshot Path: /Volumes/workspace/food_inspection/raw_data/parquet_snapshots
Snapshot Run Id  : 20260413_152456
Snapshot Date    : 2026/04/13


## Create Medallion Schemas

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
print(f"Schemas ready: {BRONZE_SCHEMA}, {SILVER_SCHEMA}, {GOLD_SCHEMA}")

Schemas ready: foodinspection_bronze, foodinspection_silver, foodinspection_gold


## Verify Raw Data Files Exist

In [0]:
try:
    files = dbutils.fs.ls(RAW_PATH)
    print("Raw data files:")
    for f in files:
        print(f"  {f.name} ({f.size / 1024 / 1024:.2f} MB)")
except Exception:
    print(f"Raw data not found at {RAW_PATH}. Please upload the CSVs.")

Raw data files:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)
  backup_Food_Inspections_20260411.csv (330.70 MB)
  parquet_snapshots/ (0.00 MB)


## Helper Utilities

In [0]:
def ensure_volume_path(path: str):
    try:
        dbutils.fs.mkdirs(path)
    except Exception as e:
        print(f"Could not create {path}: {e}")


def write_parquet_snapshot(df, base_path: str, table_name: str):
    snapshot_path = (
        f"{base_path}/{table_name}/"
        f"{SNAPSHOT_YEAR}/{SNAPSHOT_MONTH}/{SNAPSHOT_DAY}/"
        f"run_id={SNAPSHOT_RUN_ID}"
    )
    ensure_volume_path(
        f"{base_path}/{table_name}/{SNAPSHOT_YEAR}/{SNAPSHOT_MONTH}/{SNAPSHOT_DAY}"
    )
    (
        df.write
        .mode("overwrite")
        .parquet(snapshot_path)
    )
    print(f"Parquet snapshot written to: {snapshot_path}")


for path in [RAW_SNAPSHOT_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH]:
    ensure_volume_path(path)

print("Configuration ready. Variables and snapshot helpers are available via %run ./00_setup_config")

Configuration ready. Variables and snapshot helpers are available via %run ./00_setup_config
